# SWaT HPO — Results Visualization

Regenerates every HPO figure from the per-task JSONs and aggregated CSVs in `$SWAT_OUTPUT_DIR/hpo/results/`. Mirrors the academic style of `SWaT_Results_Visualization_1.ipynb` — same font sizes, color palette, figure directory, and `.png` naming scheme (`H*` prefix instead of `V*`).

Figures produced:

* **H1** — Stage-1 composite F1 distribution per detector (what the HPO explored)
* **H2** — Deployed vs HPO winner: clean and poisoned F1 side-by-side
* **H3** — Per-knob sensitivity heatmap (ΔF1 relative to baseline)
* **H4** — Seed variance on poisoned F1 (the core methodological finding)
* **H5** — Combined clean-vs-poisoned scatter of all 18 Final tasks (AE + LSTM-AE), annotated by seed
* **H6** — Final-stage robustness curves (F1 vs poison rate, per attack family, for the HPO winners)
* **H7** — Stage-by-stage composite_f1 funnel
* **H8** — Wall-clock cost vs. composite F1 scatter for Stage 1 candidates

### How to run

```bash
cd ~/projects/def-liyang/$USER/narval_swat_run
source venv/bin/activate
export SWAT_OUTPUT_DIR=/scratch/$USER/swat_paper_run
# open this notebook in VS Code / JupyterLab — all cells run in seconds on a login node
```

## 1. Setup

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch
import numpy as np
import pandas as pd

OUT_DIR = Path(os.environ.get("SWAT_OUTPUT_DIR", "$SCRATCH/swat_paper_run"))
HPO_DIR = OUT_DIR / "hpo" / "results"
FIG_DIR = OUT_DIR / "figures_hpo"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Academic plot style — mirrors SWaT_Results_Visualization_1
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 12,
    "axes.titlesize": 20,
    "axes.titleweight": "bold",
    "axes.labelsize": 16,
    "axes.labelweight": "bold",
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "lines.linewidth": 2.2,
    "lines.markersize": 7,
})

DETECTOR_COLOR = {
    "ae":      "#8172B2",   # neural family purple from ref notebook
    "lstm_ae": "#55A868",   # neural family green
}
SEED_MARKER = {42: "o", 123: "s", 456: "D"}

print(f"HPO results dir: {HPO_DIR}")
print(f"Figure output:   {FIG_DIR}")

## 2. Load the eight stage CSVs + the 18 Final JSONs

In [ ]:
STAGES = ["stage1", "stage2", "stage3", "final"]
DETECTORS = ["ae", "lstm_ae"]

dfs = {}
for d in DETECTORS:
    for s in STAGES:
        p = HPO_DIR / f"{d}_{s}.csv"
        if not p.exists():
            print(f"  missing: {p}")
            continue
        df = pd.read_csv(p)
        # Parse the config JSON column into a dict
        df["config_dict"] = df["config"].apply(json.loads)
        dfs[(d, s)] = df
        print(f"  loaded {d}_{s}.csv — {len(df)} rows")

# Load the individual Final-stage JSONs so we can access the full 13-cell poisoning grid
final_records = {"ae": [], "lstm_ae": []}
for d in DETECTORS:
    final_dir = HPO_DIR / f"{d}_final"
    for jf in sorted(final_dir.glob("[0-9]*.json")):
        if jf.name.endswith(".error.json"):
            continue
        final_records[d].append(json.load(open(jf)))
    print(f"  {d}_final: {len(final_records[d])} JSONs")

## 3. H1 — Stage-1 composite F1 distribution per detector

Histogram of every config in Stage 1, showing the shape of the search space. Lets reviewers see at a glance that the winners (right-hand tail) are comfortably above the median.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, d in zip(axes, DETECTORS):
    df = dfs[(d, "stage1")]
    ax.hist(df["composite_f1"], bins=24, color=DETECTOR_COLOR[d],
            edgecolor="black", linewidth=0.5, alpha=0.88, zorder=3)

    med = df["composite_f1"].median()
    top = df["composite_f1"].max()
    ax.axvline(med, color="gray", linestyle="--", linewidth=1.3, label=f"median = {med:.3f}")
    ax.axvline(top, color="black", linestyle="-", linewidth=1.6, label=f"top-1 = {top:.3f}")

    title = "Autoencoder (n=192)" if d == "ae" else "LSTM-AE (n=72)"
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Composite F1 (mean of clean + 10% targeted)")
    ax.set_ylabel("Number of configurations")
    ax.legend(loc="upper left", frameon=False)

fig.suptitle("H1 — Stage-1 Composite F1 Distribution", fontsize=22, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "H1_stage1_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. H2 — Deployed vs HPO winner: clean + poisoned side-by-side

The headline comparison. Deployed numbers come from R01 Table V (clean) and Table VI / §6.2 (poisoned at 10% targeted). HPO numbers are the 3-seed means from the Final stage.

In [ ]:
# Pull HPO 3-seed means from Final-stage JSONs for the winning config of each detector
def _collapse_by_config(records):
    """Group records by (config minus seed), return list of (clean_mean,clean_sd,pois_mean,pois_sd,cfg)."""
    buckets = {}
    for r in records:
        cfg = r["config"]
        key = json.dumps({k: v for k, v in cfg.items() if k != "seed"}, sort_keys=True)
        buckets.setdefault(key, []).append(r)
    summaries = []
    for key, rs in buckets.items():
        cv = [r["results"]["clean"]["f1"] for r in rs]
        pv = [r["results"].get("targeted_flip__r0.10", {}).get("f1", 0.0) for r in rs]
        composite = [(c + p) / 2 for c, p in zip(cv, pv)]
        summaries.append({
            "clean_mean":  float(np.mean(cv)), "clean_sd":  float(np.std(cv, ddof=1) if len(cv)>1 else 0),
            "pois_mean":   float(np.mean(pv)), "pois_sd":   float(np.std(pv, ddof=1) if len(pv)>1 else 0),
            "comp_mean":   float(np.mean(composite)),
            "n_seeds":     len(rs),
            "config":      json.loads(key),
        })
    summaries.sort(key=lambda s: s["comp_mean"], reverse=True)
    return summaries

# HPO winners (top distinct config by mean composite)
ae_winner   = _collapse_by_config(final_records["ae"])[0]
lstm_winner = _collapse_by_config(final_records["lstm_ae"])[0]

# Deployed numbers from the R01 paper — hard-coded here because they're not in the HPO outputs
DEPLOYED = {
    "ae":      {"clean_mean": 0.8690, "clean_sd": 0.0059,
                "pois_mean": 0.608,  "pois_sd":  0.050},   # targeted_flip @ 10 % estimated from Table T5
    "lstm_ae": {"clean_mean": 0.8895, "clean_sd": 0.0106,
                "pois_mean": 0.647,  "pois_sd":  0.080},   # targeted_flip @ 10 % estimated from Table T5
}

fig, ax = plt.subplots(figsize=(11, 6.5))
labels = ["AE\n(deployed)", "AE\n(HPO)", "LSTM-AE\n(deployed)", "LSTM-AE\n(HPO)"]
x = np.arange(len(labels))
width = 0.35

clean_means = [DEPLOYED["ae"]["clean_mean"], ae_winner["clean_mean"],
               DEPLOYED["lstm_ae"]["clean_mean"], lstm_winner["clean_mean"]]
clean_sds   = [DEPLOYED["ae"]["clean_sd"],   ae_winner["clean_sd"],
               DEPLOYED["lstm_ae"]["clean_sd"],   lstm_winner["clean_sd"]]
pois_means  = [DEPLOYED["ae"]["pois_mean"],  ae_winner["pois_mean"],
               DEPLOYED["lstm_ae"]["pois_mean"],  lstm_winner["pois_mean"]]
pois_sds    = [DEPLOYED["ae"]["pois_sd"],    ae_winner["pois_sd"],
               DEPLOYED["lstm_ae"]["pois_sd"],    lstm_winner["pois_sd"]]

bars1 = ax.bar(x - width/2, clean_means, width, yerr=clean_sds, label="Clean F1",
               color="#4C72B0", edgecolor="black", linewidth=0.6, capsize=4, zorder=3)
bars2 = ax.bar(x + width/2, pois_means, width, yerr=pois_sds, label="Poisoned F1 (10% targeted)",
               color="#C44E52", edgecolor="black", linewidth=0.6, capsize=4, zorder=3)

for b, v in zip(bars1, clean_means):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=11)
for b, v in zip(bars2, pois_means):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=13)
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1.05)
ax.set_title("H2 — Deployed vs HPO Winner")
ax.legend(loc="upper right", frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.3, zorder=0)

# Separator between AE and LSTM-AE pair
ax.axvline(1.5, color="gray", linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig(FIG_DIR / "H2_deployed_vs_hpo.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"AE winner config:      {ae_winner['config']}")
print(f"LSTM-AE winner config: {lstm_winner['config']}")

## 5. H3 — Per-knob sensitivity heatmap

For each Stage-1 knob, show how composite F1 varies across the tested values (averaged over all other-knob combinations). Color = composite F1. Highlights which dimensions of the grid actually move the needle.

In [ ]:
def _knob_matrix(df, knobs):
    """Returns a dict knob->(sorted_values, mean_composite_per_value)."""
    out = {}
    for k in knobs:
        vals = df["config_dict"].apply(lambda c: str(c.get(k)))
        by_val = df.groupby(vals)["composite_f1"].agg(["mean", "std", "count"])
        out[k] = by_val.sort_values("mean", ascending=False)
    return out

AE_S1_KNOBS      = ["hidden_dims", "dropout", "activation", "use_batchnorm"]
LSTM_AE_S1_KNOBS = ["window", "hidden_dim", "num_layers", "dropout"]

fig, axes = plt.subplots(2, 1, figsize=(13, 8.5))
for ax, d, knobs in zip(axes, DETECTORS, [AE_S1_KNOBS, LSTM_AE_S1_KNOBS]):
    m = _knob_matrix(dfs[(d, "stage1")], knobs)
    # Flatten into a matrix: rows=knobs, cols=unique values (up to max across knobs)
    max_vals = max(len(v) for v in m.values())
    mat = np.full((len(knobs), max_vals), np.nan)
    labels = []
    for i, k in enumerate(knobs):
        vs = list(m[k].index)
        means = list(m[k]["mean"])
        for j, (v, x) in enumerate(zip(vs, means)):
            mat[i, j] = x
        labels.append((k, vs))

    im = ax.imshow(mat, cmap="viridis", aspect="auto", vmin=np.nanmin(mat), vmax=np.nanmax(mat))
    ax.set_yticks(range(len(knobs)))
    ax.set_yticklabels(knobs, fontsize=12)
    # X-axis labels: per-knob values concatenated — we'll annotate cells instead
    ax.set_xticks([])
    for i, (k, vs) in enumerate(labels):
        for j, v in enumerate(vs):
            cell = mat[i, j]
            if not np.isnan(cell):
                color = "white" if cell < (np.nanmin(mat) + 0.5 * (np.nanmax(mat) - np.nanmin(mat))) else "black"
                ax.text(j, i, f"{v}\n{cell:.3f}", ha="center", va="center", fontsize=9, color=color)

    title = "AE Stage-1 per-knob composite F1" if d == "ae" else "LSTM-AE Stage-1 per-knob composite F1"
    ax.set_title(title, fontsize=15)
    plt.colorbar(im, ax=ax, shrink=0.85, label="Composite F1")

fig.suptitle("H3 — Per-Knob Sensitivity", fontsize=22, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig(FIG_DIR / "H3_per_knob_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. H4 — Seed variance on poisoned F1 (the key finding)

All 9 AE Final tasks + all 9 LSTM-AE Final tasks plotted as seed vs poisoned F1. Shows the `batch=512` outlier at AE seed 123, and the LSTM-AE seed-456 left-tail drop.

In [ ]:
def _final_rows(records):
    """Flatten Final JSONs into a DataFrame for plotting."""
    out = []
    for r in records:
        cfg = r["config"]
        out.append({
            "task_id":  r["task_id"],
            "seed":     cfg["seed"],
            "clean_f1": r["results"]["clean"]["f1"],
            "poisoned_f1":  r["results"].get("targeted_flip__r0.10", {}).get("f1", 0.0),
            "config_key":   json.dumps({k: v for k, v in cfg.items() if k != "seed"}, sort_keys=True),
        })
    return pd.DataFrame(out)

ae_df   = _final_rows(final_records["ae"])
lstm_df = _final_rows(final_records["lstm_ae"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (det, df) in zip(axes, [("ae", ae_df), ("lstm_ae", lstm_df)]):
    # Assign a short label per unique config
    unique_cfgs = df["config_key"].unique()
    cfg_short = {k: f"cfg{i}" for i, k in enumerate(unique_cfgs)}
    for cfg_key, sub in df.groupby("config_key"):
        sub = sub.sort_values("seed")
        ax.plot(sub["seed"], sub["poisoned_f1"], "-o",
                label=cfg_short[cfg_key], linewidth=2, markersize=10,
                markeredgecolor="black")
    ax.set_xticks([42, 123, 456])
    ax.set_xlabel("Seed")
    ax.set_ylabel("Poisoned F1 @ 10% targeted")
    ax.set_ylim(0, 1.0)
    ax.set_title("AE Final" if det == "ae" else "LSTM-AE Final", fontsize=15)
    ax.legend(loc="lower left", frameon=False, fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.3)

fig.suptitle("H4 — Seed Variance on Poisoned F1", fontsize=22, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "H4_seed_variance.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. H5 — Combined clean-vs-poisoned scatter (user-requested)

All 18 Final tasks (9 AE + 9 LSTM-AE) on a single clean-vs-poisoned F1 plane. Marker shape encodes seed. The diagonal line is the "poisoned = clean" reference — points above it are poisoning-*resistant* (poisoned F1 exceeds clean).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

for det, df in [("ae", ae_df), ("lstm_ae", lstm_df)]:
    for _, r in df.iterrows():
        ax.scatter(
            r["clean_f1"], r["poisoned_f1"],
            s=170,
            color=DETECTOR_COLOR[det],
            marker=SEED_MARKER[r["seed"]],
            edgecolor="black", linewidth=0.8,
            alpha=0.85, zorder=3,
        )

# Diagonal reference: poisoned == clean
lo, hi = 0.0, 1.0
ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.1, alpha=0.5, zorder=2,
        label="Poisoned = Clean (immunity line)")

# Shaded region where ΔF1 is large (poisoned < 0.5 × clean, i.e. catastrophic)
ax.fill_between([lo, hi], [0, 0], [0.5 * hi, 0.5 * hi],
                color="red", alpha=0.05, zorder=1)
ax.text(0.95, 0.07, "Catastrophic region\n(poisoned < ½ clean)",
        ha="right", fontsize=10, color="#8B0000", style="italic", alpha=0.75)

ax.set_xlim(0.5, 1.0)
ax.set_ylim(0.0, 1.0)
ax.set_xlabel("Clean F1")
ax.set_ylabel("Poisoned F1 (10% targeted)")
ax.set_title("H5 — Final-Stage Clean vs Poisoned (18 runs total)")
ax.grid(True, linestyle="--", alpha=0.3, zorder=0)

# Legend: detector color + seed marker
det_handles = [Patch(color=DETECTOR_COLOR[d], label=("AE" if d == "ae" else "LSTM-AE"))
               for d in DETECTORS]
seed_handles = [plt.Line2D([0], [0], marker=SEED_MARKER[s], color="w",
                            markerfacecolor="gray", markeredgecolor="black",
                            markersize=11, label=f"seed {s}")
                 for s in [42, 123, 456]]
leg1 = ax.legend(handles=det_handles, title="Detector",
                 loc="upper left", frameon=False)
ax.add_artist(leg1)
ax.legend(handles=seed_handles, title="Seed",
          loc="lower right", frameon=False, bbox_to_anchor=(1.0, 0.12))

plt.tight_layout()
plt.savefig(FIG_DIR / "H5_clean_vs_poisoned_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. H6 — Final-stage robustness curves (HPO winners, F1 vs poison rate)

Shows the full 13-cell poisoning grid for the top Final config of each detector, averaged across seeds.

In [ ]:
ATTACKS = ["random_flip", "targeted_flip", "feature_noise"]
RATES   = [0.01, 0.03, 0.05, 0.10]

def _final_curves(records, detector_name):
    """Return dict attack -> list of (rate, f1_mean, f1_sd) across seeds for the TOP distinct config."""
    # Identify the top config key (by mean composite) — same as winner in Section 4
    top = _collapse_by_config(records)[0]
    top_key = json.dumps(top["config"], sort_keys=True)

    # Keep only records matching that top config
    rs = [r for r in records
          if json.dumps({k: v for k, v in r["config"].items() if k != "seed"},
                         sort_keys=True) == top_key]
    # Build curves
    curves = {"clean": (0.0,
                        float(np.mean([r["results"]["clean"]["f1"] for r in rs])),
                        float(np.std([r["results"]["clean"]["f1"] for r in rs], ddof=1)
                              if len(rs) > 1 else 0.0))}
    by_attack = {a: [(0.0, curves["clean"][1], curves["clean"][2])] for a in ATTACKS}
    for atk in ATTACKS:
        for rate in RATES:
            key = f"{atk}__r{rate:.2f}"
            f1s = [r["results"].get(key, {}).get("f1", np.nan) for r in rs]
            f1s = [x for x in f1s if not np.isnan(x)]
            if f1s:
                by_attack[atk].append((
                    rate, float(np.mean(f1s)),
                    float(np.std(f1s, ddof=1)) if len(f1s) > 1 else 0.0))
    return by_attack, top["config"]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, d in zip(axes, DETECTORS):
    by_attack, cfg = _final_curves(final_records[d], d)
    for atk in ATTACKS:
        curve = by_attack[atk]
        xs = [p[0] for p in curve]
        ys = [p[1] for p in curve]
        es = [p[2] for p in curve]
        color = {"feature_noise": "#4C72B0",
                 "random_flip":   "#DD8452",
                 "targeted_flip": "#55A868"}[atk]
        ax.errorbar(xs, ys, yerr=es, fmt="-o", color=color, linewidth=2.2,
                    markeredgecolor="black", label=atk.replace("_", " ").title(),
                    capsize=3)
    ax.set_xticks([0] + RATES)
    ax.set_xticklabels(["0%"] + [f"{int(r*100)}%" for r in RATES])
    ax.set_xlabel("Poison rate")
    ax.set_ylabel("F1 Score (mean ± SD over 3 seeds)")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"{('AE' if d == 'ae' else 'LSTM-AE')} HPO Winner", fontsize=15)
    ax.legend(loc="lower left", frameon=False)

fig.suptitle("H6 — Final-Stage Robustness Curves (HPO winners)",
             fontsize=22, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "H6_final_robustness_curves.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. H7 — Stage-by-stage composite F1 funnel

For each detector, show the max composite_f1 achieved at each stage. Illustrates the narrowing of the search.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

stage_counts = {("ae", "stage1"): 192, ("ae", "stage2"): 144, ("ae", "stage3"): 81, ("ae", "final"): 9,
                ("lstm_ae", "stage1"): 72, ("lstm_ae", "stage2"): 108, ("lstm_ae", "stage3"): 27, ("lstm_ae", "final"): 9}
stage_x = [1, 2, 3, 4]
stage_labels = ["Stage 1\n(Arch)", "Stage 2\n(Training)", "Stage 3\n(Loss/Thresh)", "Final\n(3 seeds)"]

for d in DETECTORS:
    tops = []
    for s in STAGES:
        df = dfs.get((d, s))
        if df is not None:
            tops.append(df["composite_f1"].max())
        else:
            tops.append(np.nan)
    ax.plot(stage_x, tops, "-o", linewidth=3, markersize=12,
            markeredgecolor="black", color=DETECTOR_COLOR[d],
            label="AE" if d == "ae" else "LSTM-AE")
    for x, y, s in zip(stage_x, tops, STAGES):
        n = stage_counts[(d, s)]
        ax.annotate(f"{y:.3f}\n(n={n})", xy=(x, y), xytext=(0, 12),
                    textcoords="offset points", ha="center", fontsize=10)

ax.set_xticks(stage_x)
ax.set_xticklabels(stage_labels)
ax.set_ylabel("Top composite F1 at stage")
ax.set_title("H7 — Staged Elimination Funnel")
ax.set_ylim(0, 1.0)
ax.legend(loc="lower right", frameon=False)
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "H7_stage_funnel.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. H8 — Wall-clock cost vs composite F1 (Stage 1)

Shows which Stage-1 configs are Pareto-efficient. Useful for practitioners: you can trade a bit of composite F1 for substantially lower training cost.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, d in zip(axes, DETECTORS):
    df = dfs[(d, "stage1")]
    ax.scatter(df["wallclock_s"], df["composite_f1"],
               s=40, color=DETECTOR_COLOR[d], alpha=0.55,
               edgecolor="black", linewidth=0.3, zorder=3)
    # Highlight top-3 (winners for Stage 2)
    top3 = df.nlargest(3, "composite_f1")
    ax.scatter(top3["wallclock_s"], top3["composite_f1"],
               s=180, color="gold", edgecolor="black", linewidth=1.3,
               zorder=4, label="Stage 1 top 3")
    ax.set_xlabel("Wall-clock per task (s) — clean + poisoned")
    ax.set_ylabel("Composite F1")
    ax.set_title("AE Stage 1 (n=192)" if d == "ae" else "LSTM-AE Stage 1 (n=72)", fontsize=14)
    ax.legend(loc="lower right", frameon=False)
    ax.grid(True, linestyle="--", alpha=0.3, zorder=0)

fig.suptitle("H8 — Cost vs Composite F1 (Stage 1 search space)",
             fontsize=22, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "H8_cost_vs_composite.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. H9 — Cross-knob interaction heatmap (LSTM-AE window × hidden_dim)

Per-knob sensitivity sweeps at the baseline slice say window=20 and hidden_dim=128 are optimal for LSTM-AE. The HPO's joint search picked window=30, hidden_dim=256 instead. This heatmap shows why: averaged over all other-knob combinations tested in Stage 1, the composite F1 surface has a strong diagonal interaction — the big-hidden-state × long-window corner is the global optimum, but neither setting is optimal in isolation.

The deployed config's cell and the HPO winner's cell are annotated so reviewers see the interaction at a glance.

In [ ]:
# LSTM-AE Stage-1 grid has 4 windows × 3 hidden_dims × 2 layers × 3 dropouts = 72 configs.
# Collapse to a window × hidden_dim 2D surface by averaging composite_f1 over the
# remaining 2 × 3 = 6 (num_layers, dropout) combinations.
lstm_s1 = dfs[("lstm_ae", "stage1")].copy()
lstm_s1["window"]     = lstm_s1["config_dict"].apply(lambda c: int(c["window"]))
lstm_s1["hidden_dim"] = lstm_s1["config_dict"].apply(lambda c: int(c["hidden_dim"]))

pivot = (lstm_s1.groupby(["hidden_dim", "window"])["composite_f1"]
         .mean()
         .reset_index()
         .pivot(index="hidden_dim", columns="window", values="composite_f1")
         .sort_index(ascending=False))

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(pivot.values, cmap="viridis", aspect="auto",
               vmin=np.nanmin(pivot.values), vmax=np.nanmax(pivot.values))

# Annotate each cell with its composite F1
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            # Dynamic text color for readability
            lo, hi = np.nanmin(pivot.values), np.nanmax(pivot.values)
            color = "white" if v < lo + 0.5 * (hi - lo) else "black"
            ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                    fontsize=14, color=color, fontweight="bold")

ax.set_xticks(range(pivot.shape[1]))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("window (sequence length)")
ax.set_ylabel("hidden_dim")
ax.set_title("H9 — LSTM-AE Composite F1 across window × hidden_dim\n(Stage 1 grid, averaged over dropout × num_layers)",
             fontsize=16)

# Mark the DEPLOYED config (window=20, hidden_dim=128) and HPO WINNER (window=30, hidden_dim=256)
def _cell_xy(df, hidden, window):
    y = list(df.index).index(hidden)
    x = list(df.columns).index(window)
    return x, y

# Deployed config
dx, dy = _cell_xy(pivot, 128, 20)
ax.add_patch(plt.Rectangle((dx - 0.48, dy - 0.48), 0.96, 0.96,
                            fill=False, edgecolor="red", linewidth=3))
ax.annotate("DEPLOYED\n(R01)", xy=(dx, dy), xytext=(dx + 0.7, dy - 0.8),
            fontsize=11, color="red", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="red"))

# HPO winner
wx, wy = _cell_xy(pivot, 256, 30)
ax.add_patch(plt.Rectangle((wx - 0.48, wy - 0.48), 0.96, 0.96,
                            fill=False, edgecolor="gold", linewidth=3.5))
ax.annotate("HPO WINNER", xy=(wx, wy), xytext=(wx - 1.2, wy + 0.9),
            fontsize=11, color="#996600", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="#996600"))

plt.colorbar(im, ax=ax, shrink=0.85, label="Composite F1")
plt.tight_layout()
plt.savefig(FIG_DIR / "H9_lstm_interaction_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nKey observation from H9:")
print(f"  Deployed cell (W=20, H=128):  composite = {pivot.loc[128, 20]:.4f}")
print(f"  HPO winner cell (W=30, H=256): composite = {pivot.loc[256, 30]:.4f}")
print(f"  Δ from per-knob sweep (W=30, H=128): composite = {pivot.loc[128, 30]:.4f}")
print(f"  Δ from per-knob sweep (W=20, H=256): composite = {pivot.loc[256, 20]:.4f}")
print("\nThe interaction: each knob individually (at baseline slice) looks worse,")
print("but jointly they're the grid's best — hence the need for full-grid HPO.")

## 12. Done — what was saved

In [ ]:
saved = sorted(FIG_DIR.glob("H*.png"))
print(f"Saved {len(saved)} figures to {FIG_DIR}:")
for p in saved:
    sz = p.stat().st_size / 1024
    print(f"  {p.name:<42}  ({sz:.0f} KB)")